# RP2

In [1]:
#pip install loompy

In [3]:
import scanpy as sc
import pandas as pd
import anndata as ad
import os

In [ ]:
# Load 10X data directly
samples = ("GSM5293845_A1", "GSM5293846_A2", "GSM5293847_A3", "GSM5293848_A7", "GSM5293849_A8")
for sample_name in samples:
    adata = sc.read_10x_mtx(
        '../data/',  
        prefix = f'{sample_name}.'
    )

# Make variable names unique
#adata.var_names_unique()

# Add sample information
#adata.obs['sample'] = 'placenta_snRNA'
#adata.obs['tissue'] = 'placenta'

# Save as H5AD
#adata.write('placenta_snRNA.h5ad')

#adata = sc.read_mtx()

In [6]:
adata = sc.read_h5ad("../data/vento-tormo-et-al-2018/vento18_10x.processed.h5ad")
adata

AnnData object with n_obs × n_vars = 59705 × 25875
    obs: 'CellType', 'Stage', 'n_counts', 'log1p_n_counts', 'n_genes', 'log1p_n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'percent_top50', 'Location'
    var: 'gene_ids', 'mito', 'ribo', 'hb', 'n_counts', 'n_cells', 'n_genes', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    obsm: 'X_umap_hm'

In [9]:
adata.obs['Location'].value_counts()

Location
Decidua     34072
Placenta    16075
Blood        9558
Name: count, dtype: int64

In [18]:
adata.obs['Stage'].value_counts()

Stage
8 + 2 LMP (6 + 2 PCW)      17140
12 + 1 LMP (12 + 1 PCW)    12546
9+2 GW / LMP (7 PCW)        9335
12+2 LMP(10+2 PCW)          7746
9+4 LMP(7+4 PCW)            6426
9 + 2GW (7 + 2 PCW)         4663
6 GW / LMP (4 PCW)          1849
Name: count, dtype: int64

In [13]:
adata1 = sc.read_h5ad("../data/arutyunyan-et-al-2023/adata_all_donors_all_cell_states_raw_counts_in_raw_normlog_counts_in_X_for_download_UPD_20230307.h5ad")
adata1

AnnData object with n_obs × n_vars = 325665 × 30800
    obs: 'batch', 'cell_type', 'celltype_predictions', 'coarse_annot', 'dataset', 'dev_age', 'donor', 'number_of_individuals_multiplexed', 'origin_M_F', 'phase', 'sample', 'technology', 'tissue', 'n_counts', 'barcode_sample_copy'
    var: 'gene_ids-0', 'feature_types-0', 'genome-0', 'n_cells-0', 'gene_ids-1', 'feature_types-1', 'genome-1', 'n_cells-1', 'gene_ids-10', 'feature_types-10', 'genome-10', 'n_cells-10', 'gene_ids-11', 'feature_types-11', 'genome-11', 'n_cells-11', 'gene_ids-12', 'feature_types-12', 'genome-12', 'n_cells-12', 'gene_ids-13', 'feature_types-13', 'genome-13', 'n_cells-13', 'gene_ids-14', 'feature_types-14', 'genome-14', 'n_cells-14', 'gene_ids-15', 'feature_types-15', 'genome-15', 'n_cells-15', 'gene_ids-16', 'feature_types-16', 'genome-16', 'n_cells-16', 'gene_ids-17', 'feature_types-17', 'genome-17', 'n_cells-17', 'gene_ids-18', 'feature_types-18', 'genome-18', 'n_cells-18', 'gene_ids-19', 'feature_types-19', 

In [15]:
adata1.obs['tissue'].value_counts()

tissue
decidua                        107110
placenta                        66499
decidua_placenta_myometrium     66038
decidua_myometrium              32674
decidua_immune                  22691
decidua_non_immune              19664
blood                           10989
Name: count, dtype: int64

In [20]:
adata1.obs['dev_age'].value_counts()

dev_age
6_PCW        75604
8-9_PCW      66038
8_PCW        64644
9_PCW        38324
10_PCW       25170
12_PCW       19299
7-8_PCW      16833
5_PCW        12874
12-13_PCW     3558
4-5_PCW       3321
Name: count, dtype: int64

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Set scanpy settings
sc.settings.verbosity = 3  # verbosity level
sc.settings.set_figure_params(dpi=80, facecolor='white')

def load_10x_sample(sample_path, sample_name):
    """
    Load a single 10X sample and return AnnData object
    """
    print(f"Loading sample: {sample_name}")
    
    # Read the 10X data
    adata = sc.read_10x_mtx(
        sample_path,  # Path to the directory containing matrix.mtx file
        var_names='gene_symbols',  # use gene symbols for gene names (variables names)
        cache=True,  # write a cache file for faster subsequent reading
        gex_only=True  # only keep 'Gene Expression' data
    )
    
    # Make variable names unique (in case there are duplicate gene names)
    adata.var_names_unique()
    
    # Add sample information
    adata.obs['sample'] = sample_name
    
    return adata

def process_multiple_samples(data_dir):
    """
    Process multiple 10X samples and concatenate them
    """
    data_path = Path(data_dir)
    
    # Get all sample directories (assuming each sample has its own folder)
    # Based on your file structure, it looks like files are directly in the folder
    # We'll group them by sample ID (everything before the dot)
    
    files = list(data_path.glob("*.mtx.gz"))
    
    if not files:
        print("No .mtx.gz files found. Looking for uncompressed files...")
        files = list(data_path.glob("*.mtx"))
    
    if not files:
        raise FileNotFoundError("No matrix files found in the specified directory")
    
    # Extract sample names from matrix files
    sample_names = []
    for file in files:
        # Extract sample name (e.g., GSM5293845_A1 from GSM5293845_A1.matrix.mtx.gz)
        sample_name = file.stem.replace('.matrix', '').replace('.mtx', '')
        sample_names.append(sample_name)
    
    print(f"Found {len(sample_names)} samples: {sample_names}")
    
    # Load each sample
    adatas = []
    for sample_name in sample_names:
        try:
            # For your file structure, we need to create temporary directory structure
            # that scanpy expects, or read files individually
            
            # Method 1: Read files individually and create AnnData object
            matrix_file = data_path / f"{sample_name}.matrix.mtx.gz"
            barcodes_file = data_path / f"{sample_name}.barcodes.tsv.gz"
            features_file = data_path / f"{sample_name}.features.tsv.gz"
            
            # Check if files exist
            if not all([matrix_file.exists(), barcodes_file.exists(), features_file.exists()]):
                print(f"Warning: Missing files for sample {sample_name}, skipping...")
                continue
            
            # Read the individual files
            from scipy.io import mmread
            import gzip
            
            # Read matrix
            with gzip.open(matrix_file, 'rb') as f:
                matrix = mmread(f).T.tocsr()  # Transpose to get cells x genes
            
            # Read barcodes
            barcodes = pd.read_csv(barcodes_file, sep='\t', header=None, names=['barcode'])
            
            # Read features
            features = pd.read_csv(features_file, sep='\t', header=None, 
                                 names=['gene_id', 'gene_symbol', 'feature_type'])
            
            # Sanity check: matrix dimensions should match barcodes and features
            print(f"  Matrix shape: {matrix.shape} (cells x genes)")
            print(f"  Barcodes: {len(barcodes)} entries")
            print(f"  Features: {len(features)} entries")
            
            if matrix.shape[0] != len(barcodes):
                print(f"  ⚠️  WARNING: Matrix rows ({matrix.shape[0]}) don't match barcodes ({len(barcodes)})")
                print(f"  This might indicate a matrix orientation issue")
            
            if matrix.shape[1] != len(features):
                print(f"  ⚠️  WARNING: Matrix columns ({matrix.shape[1]}) don't match features ({len(features)})")
                print(f"  This might indicate a matrix orientation issue")
            
            # Make gene symbols unique before creating AnnData object
            gene_symbols = features['gene_symbol'].values
            unique_gene_symbols = pd.Index(gene_symbols).unique()
            
            # If there are duplicates, make them unique
            if len(gene_symbols) != len(unique_gene_symbols):
                print(f"  Found {len(gene_symbols) - len(unique_gene_symbols)} duplicate gene symbols, making unique...")
                gene_symbols = pd.Series(gene_symbols).values
                # Use pandas to make names unique
                gene_symbols_series = pd.Series(gene_symbols)
                duplicated_mask = gene_symbols_series.duplicated(keep=False)
                
                for i, is_dup in enumerate(duplicated_mask):
                    if is_dup:
                        # Count how many times this gene symbol has appeared so far
                        current_symbol = gene_symbols_series.iloc[i]
                        count = (gene_symbols_series.iloc[:i+1] == current_symbol).sum()
                        if count > 1:
                            gene_symbols[i] = f"{current_symbol}-{count-1}"
            
            # Create AnnData object with unique gene names
            adata = sc.AnnData(
                X=matrix,
                obs=pd.DataFrame(index=barcodes['barcode']),
                var=pd.DataFrame(index=gene_symbols)
            )
            
            # Add additional information
            adata.var['gene_ids'] = features['gene_id'].values
            adata.var['feature_types'] = features['feature_type'].values
            adata.var['original_gene_symbols'] = features['gene_symbol'].values  # Keep original names
            adata.obs['sample'] = sample_name
            
            # Add sample prefix to cell barcodes to make them unique
            adata.obs.index = [f"{sample_name}_{barcode}" for barcode in adata.obs.index]
            
            adatas.append(adata)
            print(f"Loaded {sample_name}: {adata.n_obs} cells, {adata.n_vars} genes")
            
        except Exception as e:
            print(f"Error loading sample {sample_name}: {str(e)}")
            continue
    
    if not adatas:
        raise ValueError("No samples were successfully loaded")
    
    # Concatenate all samples
    print("Concatenating samples...")
    
    # First, let's check if all samples have the same genes
    print("Checking gene consistency across samples...")
    all_genes = [set(adata.var_names) for adata in adatas]
    common_genes = set.intersection(*all_genes) if all_genes else set()
    all_unique_genes = set.union(*all_genes) if all_genes else set()
    
    print(f"Common genes across all samples: {len(common_genes)}")
    print(f"Total unique genes across all samples: {len(all_unique_genes)}")
    
    # Check for any remaining duplicate indices in cell names
    print("Checking for duplicate cell indices...")
    all_cell_indices = []
    for i, adata in enumerate(adatas):
        print(f"  Sample {i}: {len(adata.obs.index)} cells")
        # Check for duplicates within this sample
        if adata.obs.index.duplicated().any():
            print(f"    Warning: Sample {i} has duplicate cell indices")
        all_cell_indices.extend(adata.obs.index.tolist())
    
    if len(all_cell_indices) != len(set(all_cell_indices)):
        print("  Found duplicate cell indices across samples - this shouldn't happen with our prefixing")
    
    # Check for duplicate gene indices
    print("Checking for duplicate gene indices...")
    for i, adata in enumerate(adatas):
        if adata.var.index.duplicated().any():
            print(f"  Warning: Sample {i} has duplicate gene indices")
            # Force make them unique using pandas make_unique method
            duplicated_genes = adata.var.index.duplicated(keep=False)
            print(f"    Found {duplicated_genes.sum()} duplicate gene entries")
            
            # Use pandas to make the index unique
            new_index = []
            gene_counts = {}
            
            for gene_name in adata.var.index:
                if gene_name in gene_counts:
                    gene_counts[gene_name] += 1
                    new_index.append(f"{gene_name}_{gene_counts[gene_name]}")
                else:
                    gene_counts[gene_name] = 0
                    new_index.append(gene_name)
            
            adata.var.index = pd.Index(new_index)
            print(f"    Made gene indices unique for sample {i}")
    
    # Try concatenation with different parameters
    try:
        print("Attempting concatenation with join='outer'...")
        adata_combined = sc.concat(adatas, join='outer')
    except Exception as e:
        print(f"Failed with join='outer': {e}")
        try:
            print("Attempting concatenation with join='inner'...")
            adata_combined = sc.concat(adatas, join='inner')
        except Exception as e2:
            print(f"Failed with join='inner': {e2}")
            print("Trying manual concatenation approach...")
            
            # Manual concatenation approach
            # Find common genes across all samples
            common_gene_list = list(common_genes)
            common_gene_list.sort()  # Sort for consistency
            
            # Subset each sample to common genes only
            adatas_subset = []
            for adata in adatas:
                adata_subset = adata[:, common_gene_list].copy()
                adatas_subset.append(adata_subset)
            
            # Now try concatenation again
            adata_combined = sc.concat(adatas_subset, join='outer')
    
    print(f"Combined dataset: {adata_combined.n_obs} cells, {adata_combined.n_vars} genes")
    print(f"Samples: {adata_combined.obs['sample'].value_counts()}")
    
    # Detailed diagnosis
    print("\n=== DETAILED DIAGNOSIS ===")
    print("Individual sample cell counts:")
    total_expected = 0
    for i, adata in enumerate(adatas):
        sample_name = adata.obs['sample'].iloc[0]
        n_cells = adata.n_obs
        print(f"  {sample_name}: {n_cells:,} cells")
        total_expected += n_cells
    
    print(f"\nExpected total: {total_expected:,} cells")
    print(f"Actual combined: {adata_combined.n_obs:,} cells")
    print(f"Difference: {adata_combined.n_obs - total_expected:,} cells")
    
    if adata_combined.n_obs != total_expected:
        print("\n⚠️  WARNING: Cell count mismatch detected!")
        print("This suggests an issue with concatenation or duplicate counting.")
    
    # Check for any obvious issues
    print(f"\nUnique samples in combined data: {adata_combined.obs['sample'].nunique()}")
    print(f"Expected number of samples: {len(adatas)}")
    
    # Check if there are any duplicate cell barcodes
    duplicate_barcodes = adata_combined.obs.index.duplicated().sum()
    if duplicate_barcodes > 0:
        print(f"⚠️  Found {duplicate_barcodes} duplicate cell barcodes!")
    else:
        print("✓ No duplicate cell barcodes found")
    
    # Show sample distribution
    print(f"\nSample distribution in combined dataset:")
    sample_counts = adata_combined.obs['sample'].value_counts().sort_index()
    for sample, count in sample_counts.items():
        print(f"  {sample}: {count:,} cells")
    
    return adata_combined

def main():
    # Set your data directory path
    data_directory = "../data/uhm-et-al-2025/"  # Update this path
    
    # Process all samples
    adata = process_multiple_samples(data_directory)
    
    # Basic information
    print("\nDataset overview:")
    print(f"Number of cells: {adata.n_obs}")
    print(f"Number of genes: {adata.n_vars}")
    print(f"Samples: {list(adata.obs['sample'].unique())}")
    
    # Optional: Add some basic metadata
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.var['n_cells'] = (adata.X > 0).sum(axis=0).A1
    
    # Save to h5ad format
    output_file = "combined_dataset.h5ad"
    print(f"\nSaving to {output_file}...")
    adata.write(output_file)
    
    print("Done! Your data has been saved to combined_dataset.h5ad")
    
    return adata

# Alternative function if you want to process samples in subdirectories
def load_samples_from_subdirs(base_dir):
    """
    Use this if each sample is in its own subdirectory with standard 10X structure
    """
    base_path = Path(base_dir)
    sample_dirs = [d for d in base_path.iterdir() if d.is_dir()]
    
    adatas = []
    for sample_dir in sample_dirs:
        sample_name = sample_dir.name
        adata = load_10x_sample(sample_dir, sample_name)
        adatas.append(adata)
    
    return sc.concat(adatas, join='outer')

if __name__ == "__main__":
    # Run the main function
    adata = main()

Found 30 samples: ['GSM5293845_A1', 'GSM5293846_A2', 'GSM5293847_A3', 'GSM5293848_A7', 'GSM5293849_A8', 'GSM5293850_B1', 'GSM5293851_B2', 'GSM5293852_B4', 'GSM5293853_B5', 'GSM5293854_B8', 'GSM5293855_C1', 'GSM5293856_C2', 'GSM5293857_C3', 'GSM5293858_C4', 'GSM5293859_C7', 'GSM5293860_D1', 'GSM5293861_D8', 'GSM5293862_D9', 'GSM5293863_D10', 'GSM5293864_D13', 'GSM5293865_E1', 'GSM5293866_E2', 'GSM5293867_E3', 'GSM5293868_E4', 'GSM5293869_E5', 'GSM5293870_N1', 'GSM5293871_N2', 'GSM5293872_N3', 'GSM5293873_N4', 'GSM5293874_N5']
  Matrix shape: (15019, 32738) (cells x genes)
  Barcodes: 15019 entries
  Features: 32738 entries
  Found 95 duplicate gene symbols, making unique...


c:\Users\doram\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM5293845_A1: 15019 cells, 32738 genes
  Matrix shape: (26021, 32738) (cells x genes)
  Barcodes: 26021 entries
  Features: 32738 entries
  Found 95 duplicate gene symbols, making unique...


c:\Users\doram\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM5293846_A2: 26021 cells, 32738 genes
  Matrix shape: (3720, 32738) (cells x genes)
  Barcodes: 3720 entries
  Features: 32738 entries
  Found 95 duplicate gene symbols, making unique...


...

Loaded GSM5293872_N3: 24887 cells, 32738 genes
  Matrix shape: (8887, 32738) (cells x genes)
  Barcodes: 8887 entries
  Features: 32738 entries
  Found 95 duplicate gene symbols, making unique...


c:\Users\doram\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM5293873_N4: 8887 cells, 32738 genes
  Matrix shape: (21018, 32738) (cells x genes)
  Barcodes: 21018 entries
  Features: 32738 entries
  Found 95 duplicate gene symbols, making unique...


c:\Users\doram\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Loaded GSM5293874_N5: 21018 cells, 32738 genes
Concatenating samples...
Checking gene consistency across samples...
Common genes across all samples: 32734
Total unique genes across all samples: 32734
Checking for duplicate cell indices...
  Sample 0: 15019 cells
  Sample 1: 26021 cells
  Sample 2: 3720 cells
  Sample 3: 24074 cells
  Sample 4: 13601 cells
  Sample 5: 13157 cells
  Sample 6: 27531 cells
  Sample 7: 12987 cells
  Sample 8: 16668 cells
  Sample 9: 21584 cells
  Sample 10: 13684 cells
  Sample 11: 17496 cells
  Sample 12: 18279 cells
  Sample 13: 14532 cells
  Sample 14: 12061 cells
  Sample 15: 27056 cells
  Sample 16: 19997 cells
  Sample 17: 29907 cells
  Sample 18: 14988 cells
  Sample 19: 22489 cells
  Sample 20: 15607 cells
  Sample 21: 18184 cells
  Sample 22: 26644 cells
  Sample 23: 20304 cells
  Sample 24: 9875 cells
  Sample 25: 26171 cells
  Sample 26: 21480 cells
  Sample 27: 24887 cells
  Sample 28: 8887 cells
  Sample 29: 21018 cells
Checking for duplicate g

In [3]:
adata2 = sc.read_h5ad("combined_dataset.h5ad")
adata2

AnnData object with n_obs × n_vars = 557908 × 32738
    obs: 'sample', 'n_genes', 'n_counts'
    var: 'n_cells'